In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import pickle
from sklearn.metrics import mean_squared_error as mse, r2_score as r2

In [2]:
X_train = pd.read_excel('Data/Train test split/X_train.xlsx')
X_test = pd.read_excel('Data/Train test split/X_test.xlsx')
y_train = pd.read_excel('Data/Train test split/y_train.xlsx')
y_test = pd.read_excel('Data/Train test split/y_test.xlsx')

In [8]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern

# 1. Instantiate the Kernel
# We use separate length scales for the 3 variables: length_scale=[l1, l2, l3]
# Matern(nu=2.5) is usually superior to RBF for physical systems as it relaxes 
# the assumption of infinite differentiability.
kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0, 1.0, 1.0], 
                                       length_scale_bounds=(1e-2, 1e3), 
                                       nu=2.5)

# 2. Instantiate the GPR
# Since Ansys is deterministic, we set 'alpha' (the noise variance) very low.
# n_restarts_optimizer handles local minima in the Marginal Likelihood optimization.
gpr = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-10, 
    n_restarts_optimizer=10,
    normalize_y=True
)

# 3. Fit the model on your grid data (increments of 5)
gpr.fit(X_train, y_train)

# 4. Predict intermediate points (X_test has fractional values)
y_pred, y_std = gpr.predict(X_test, return_std=True)

c:\Users\rajee\miniconda3\envs\torchenv\lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 22 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
c:\Users\rajee\miniconda3\envs\torchenv\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [9]:
y_pred

array([ 607.70466106,  179.42732318,  171.70731527,  256.48761831,
        488.21116147,  213.90472219, 1039.51839293,  472.06190499,
        224.81986691,  344.30341036,  379.48837453,  312.59106068,
        171.31349853,  512.39821697,  824.2213342 ,  194.68736877,
        682.15711782,  382.89036071,  236.35538871,  335.94851495,
        220.55943275,  443.40798824,  178.22166824,  240.03367352,
        446.43020947,  531.44598615, 1567.8252757 ,  221.59729095,
        278.01734386,  282.71376379,  286.30217852,  186.5022801 ,
        661.30095401,  156.20996011,  410.82885243,  378.45123678,
        550.24682562,  330.5247765 ,  227.68267192,  159.96378169,
        275.32240521,  311.85248287,  224.25454067,  363.24667039,
        486.46981713,  268.88668573,  471.1847692 ,  234.83809373,
        396.27098831,  268.98696904,  909.2119481 ,  281.76834418,
        226.46900727,  205.33511906,  169.03018506,  198.22088551,
        247.21646789,  222.83908342,  184.2943702 ,  411.26950

In [ ]:
# with open('Models/gpr.pkl', 'wb') as f:
#     pickle.dump(gpr, f)

In [3]:
with open('Models/gpr.pkl', 'rb') as f:
    gpr = pickle.load(f)

In [7]:
y_pred, y_std = gpr.predict(X_test, return_std=True)

In [5]:
df = pd.read_excel('Data/Data_final.xlsx')
df.dropna(inplace = True)
X = df.drop(columns = 'Fr')
df['gpr pred'] = gpr.predict(X)

# df.to_excel('gpr pred.xlsx', index = False)

In [8]:
mse(y_test,y_pred), r2(y_test,y_pred)

(0.34926607795742864, 0.9999977139602879)

In [9]:
testdf = pd.DataFrame(
    [
        [30,33,38],
        [30,43,48],
        [30,53,58],
        [30,63,68],
        [30,73,78],
        [30,83,88],
        [30,93,98],
        [30,103,108],
        [30,113,118],
        [30,123,128],
        [30,133,138],
        [30,143,148],
        [50,33,39],
        [50,42,51],
        [50,64,69],
        [50,76,83],
        [50,87,94],
        [50,99,108],
        [50,113,119],
        [50,126,131],
        [50,137,142],
        [50,143,149]

    ],
    columns = ['CuP', 'W', 'L']
)

In [11]:
gpr.predict(testdf)

array([2163.68413587, 1367.089143  ,  933.56524744,  676.71953795,
        511.74726464,  400.24077235,  321.38856516,  263.54255549,
        219.89391553,  186.21243065,  159.70680201,  138.40964711,
       2160.00682881, 1327.5756803 ,  675.69058548,  474.05808431,
        366.92488104,  280.51231222,  224.0739287 ,  182.63991728,
        154.91525645,  141.29612939])